**© Copyright AIDENTIFY. All rights reserved.**

# Part 1 | Session 07: HuggingFace Transformers & TRL 기본 사용법

---

### 📋 학습 목표

본 세션은 Session 06(파인튜닝 개념)을 마치고 Session 10(본격 SFT 실습)에 들어가기 전에, **실습에 사용할 두 라이브러리를 가볍게 둘러보는 오리엔테이션** 입니다.

**Part A — 🤗 HuggingFace Transformers 기본**
- 🔹 `AutoTokenizer` / `AutoModelForCausalLM`로 모델·토크나이저 로딩
- 🔹 `model.generate()`로 직접 추론 (텍스트 생성)
- 🔹 `pipeline()` API로 간편 추론
- 🔹 토크나이저 동작과 chat template 이해

**Part B — 🦾 TRL 라이브러리 기본**
- 🔹 TRL(Transformer Reinforcement Learning)이 제공하는 Trainer 종류 (`SFTTrainer`, `DPOTrainer`, `GRPOTrainer` 등) 개요
- 🔹 `TrainingArguments` / `SFTConfig` 핵심 파라미터 의미
- 🔹 `SFTTrainer`의 인터페이스 미리보기 (코드만 둘러봄, 실제 학습은 Session 12에서)

### 📦 필요 라이브러리

```
transformers, trl, torch, accelerate
```

### ⏱️ 예상 소요 시간: 약 60분

### 🖥️ 요구 사양
- GPU: RTX 4060 (8GB VRAM) 이상 권장 (CPU 추론도 가능하지만 느림)
- 모델: **Qwen2.5-1.5B-Instruct** (본 과정 표준 실습 모델)

> 💡 본 노트북은 **도구 사용법 위주**입니다. 본격적인 SFT 학습 루프는 Session 12에서 다룹니다.

---

In [1]:
# uv add trl

In [2]:
# 환경 점검 + 라이브러리 임포트
import torch, gc

print(f"PyTorch: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB")

# 패키지 버전 확인
import transformers, accelerate
try:
    import trl
    trl_ver = trl.__version__
except ImportError:
    trl_ver = "❌ 미설치 (uv add trl)"

print(f"\ntransformers: {transformers.__version__}")
print(f"accelerate:   {accelerate.__version__}")
print(f"trl:          {trl_ver}")


def print_gpu_memory(tag=""):
    """현재 GPU 메모리 사용량 출력 (CPU 환경에서는 안내만 출력)"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3   # GB
        reserved  = torch.cuda.memory_reserved()  / 1024**3   # GB
        print(f"🖥️  [{tag}] GPU 메모리 — 사용 중: {allocated:.2f} GB | 예약됨: {reserved:.2f} GB")
    else:
        print(f"🖥️  [{tag}] CUDA 사용 불가 (CPU 모드) — GPU 메모리 측정 생략")

PyTorch: 2.8.0+cu128
CUDA 사용 가능: True
GPU: NVIDIA GeForce RTX 3090
VRAM: 23.6GB

transformers: 4.52.0
accelerate:   1.15.0
trl:          0.19.1


---

# 🤗 Part A — HuggingFace Transformers 기본

---

## 1️⃣ AutoModel / AutoTokenizer로 모델 로딩

`Auto*` 클래스는 모델 이름만 주면 적절한 아키텍처를 자동으로 선택해 로딩합니다.

| 클래스 | 용도 |
|--------|------|
| `AutoTokenizer` | 토크나이저 자동 로딩 |
| `AutoModelForCausalLM` | 디코더-only LM (GPT, LLaMA, Qwen 등) |
| `AutoModelForSeq2SeqLM` | 인코더-디코더 (T5, BART 등) |
| `AutoModelForSequenceClassification` | 분류 |

---

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"📥 모델 로딩 시작: {MODEL_NAME}")
print("⏳ 처음 실행 시 다운로드에 시간이 걸립니다...")
print_gpu_memory("로딩 전")

start_time = time.time()

# 1. 토크나이저 로딩
print("\n🔤 토크나이저 로딩 중...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"  ✅ 토크나이저 로딩 완료")
print(f"  📊 어휘 크기: {tokenizer.vocab_size:,}")

# 2. 모델 로딩 (FP16)
print("\n🧠 모델 로딩 중 (FP16)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

elapsed = time.time() - start_time
print(f"  ✅ 모델 로딩 완료! ({elapsed:.1f}초)")
print_gpu_memory("모델 로딩 후")

# 모델 정보 출력
total_params = sum(p.numel() for p in model.parameters())
print(f"\n📊 모델 정보:")
print(f"  파라미터 수: {total_params:,} ({total_params/1e9:.2f}B)")
print(f"  모델 타입: {model.config.model_type}")
print(f"  히든 크기: {model.config.hidden_size}")
print(f"  레이어 수: {model.config.num_hidden_layers}")
print(f"  어텐션 헤드 수: {model.config.num_attention_heads}")

📥 모델 로딩 시작: Qwen/Qwen2.5-1.5B-Instruct
⏳ 처음 실행 시 다운로드에 시간이 걸립니다...
🖥️  [로딩 전] GPU 메모리 — 사용 중: 0.00 GB | 예약됨: 0.00 GB

🔤 토크나이저 로딩 중...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  ✅ 토크나이저 로딩 완료
  📊 어휘 크기: 151,643

🧠 모델 로딩 중 (FP16)...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  ✅ 모델 로딩 완료! (49.3초)
🖥️  [모델 로딩 후] GPU 메모리 — 사용 중: 0.00 GB | 예약됨: 0.00 GB

📊 모델 정보:
  파라미터 수: 1,543,714,304 (1.54B)
  모델 타입: qwen2
  히든 크기: 1536
  레이어 수: 28
  어텐션 헤드 수: 12


---

## 2️⃣ 토크나이저 동작 이해

`tokenizer.encode()` / `tokenizer.decode()` / `tokenizer.apply_chat_template()` 을 직접 호출해 동작을 확인합니다.

---

In [4]:
# 🔤 토크나이저 동작 이해하기
text = "인공지능은 미래를 바꿀 것입니다."

print("🔤 토크나이저 동작 이해")
print("=" * 50)
print(f"원본 텍스트: {text}")

# 인코딩
tokens = tokenizer.encode(text)
print(f"\n📊 토큰 ID: {tokens}")
print(f"📊 토큰 수: {len(tokens)}")

# 개별 토큰 확인
print(f"\n🔍 개별 토큰 분석:")
for i, token_id in enumerate(tokens):
    token_str = tokenizer.decode([token_id])
    print(f"  {i}: ID={token_id:6d} → '{token_str}'")

# 디코딩
decoded = tokenizer.decode(tokens)
print(f"\n🔄 디코딩 결과: {decoded}")

🔤 토크나이저 동작 이해
원본 텍스트: 인공지능은 미래를 바꿀 것입니다.

📊 토큰 ID: [31328, 78125, 21329, 66019, 33704, 143005, 18411, 81718, 144447, 130882, 13]
📊 토큰 수: 11

🔍 개별 토큰 분석:
  0: ID= 31328 → '인'
  1: ID= 78125 → '공'
  2: ID= 21329 → '지'
  3: ID= 66019 → '능'
  4: ID= 33704 → '은'
  5: ID=143005 → ' 미래'
  6: ID= 18411 → '를'
  7: ID= 81718 → ' 바'
  8: ID=144447 → '꿀'
  9: ID=130882 → ' 것입니다'
  10: ID=    13 → '.'

🔄 디코딩 결과: 인공지능은 미래를 바꿀 것입니다.


---

## 3️⃣ model.generate()로 직접 추론

Chat 모델은 `apply_chat_template`로 messages를 텍스트로 만든 뒤 `model.generate()`를 호출합니다.

```python
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
output_ids = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
response = tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
```

---

In [5]:
# 🧠 Chat 템플릿을 사용한 추론
print("🧠 Chat 템플릿을 사용한 추론")
print("=" * 50)

messages = [
    {"role": "system", "content": "당신은 유능한 AI 어시스턴트입니다. 간결하게 답변해주세요."},
    {"role": "user", "content": "파이썬의 장점 3가지를 알려주세요."}
]

# Chat 템플릿 적용
text_input = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
print(f"📝 적용된 템플릿:\n{text_input}")
print(f"\n{'=' * 50}")

# 토큰화 및 추론
inputs = tokenizer(text_input, return_tensors="pt").to(model.device)
print(f"\n📊 입력 토큰 수: {inputs['input_ids'].shape[1]}")

start_time = time.time()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )

elapsed = time.time() - start_time

# 생성된 토큰만 디코딩
generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
response = tokenizer.decode(generated_ids, skip_special_tokens=True)

print(f"\n💬 모델 응답:\n{response}")
print(f"\n⏱️ 소요 시간: {elapsed:.2f}초")
print(f"📊 생성 토큰 수: {len(generated_ids)}")
print(f"⚡ 처리 속도: {len(generated_ids)/elapsed:.1f} tokens/sec")
print_gpu_memory("추론 후")

🧠 Chat 템플릿을 사용한 추론
📝 적용된 템플릿:
<|im_start|>system
당신은 유능한 AI 어시스턴트입니다. 간결하게 답변해주세요.<|im_end|>
<|im_start|>user
파이썬의 장점 3가지를 알려주세요.<|im_end|>
<|im_start|>assistant



📊 입력 토큰 수: 47



💬 모델 응답:
1. 가독성: 파이썬 코드는 읽기 쉽게 작성할 수 있어 개발자들의 이해와 편의성이 높습니다.
2. 확장성: 다양한 패키지와 라이브러리를 제공하여 프로젝트를 확장하고 개선할 수 있습니다.
3. 효율성: 동적 타입 시스템과 객체 지향 프로그래밍(OOP) 기반으로 코드는 실행 속도가 빠릅니다.

⏱️ 소요 시간: 57.75초
📊 생성 토큰 수: 111
⚡ 처리 속도: 1.9 tokens/sec
🖥️  [추론 후] GPU 메모리 — 사용 중: 0.00 GB | 예약됨: 0.00 GB


---

## 4️⃣ Pipeline API — 간편 추론

`pipeline()` 은 토큰화 / 추론 / 디코딩을 한 함수로 묶어 줍니다. 빠른 프로토타이핑에 유용.

---

In [6]:
from transformers import pipeline

# 🔧 이미 로딩된 모델로 Pipeline 생성
print("🔧 Pipeline 생성 중...")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

print("✅ Pipeline 생성 완료!")

# Pipeline으로 추론
print("\n💬 Pipeline 추론 테스트")
print("=" * 50)

messages = [
    {"role": "system", "content": "간결하게 한국어로 답변하세요."},
    {"role": "user", "content": "딥러닝이란 무엇인가요?"}
]

start_time = time.time()

result = pipe(
    messages,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True, # 만약 do_sample=False로 설정하면 가장 확률이 높은 토큰만 선택하여 더 결정론적인 출력을 생성합니다 (Greedy)
)

elapsed = time.time() - start_time

response_text = result[0]["generated_text"][-1]["content"]
print(f"📝 응답:\n{response_text}")
print(f"\n⏱️ 소요 시간: {elapsed:.2f}초")

Device set to use cuda:0


🔧 Pipeline 생성 중...
✅ Pipeline 생성 완료!

💬 Pipeline 추론 테스트
📝 응답:
딥러닝은 인공지능에서 가장 중요한 기술 중 하나입니다. 이는 컴퓨터가 데이터를 분석하고 학습하여 새로운 정보를 생성하는 능력을 가진 시스템을 말합니다. 딥러닝의 핵심은 머신 러닝과 유사하지만, 더 복잡한 모델을 사용하여 더 높은 정확도와 효율성을 제공한다는 점입니다.

⏱️ 소요 시간: 3.81초


In [7]:
# 📝 다양한 질문 테스트
test_questions = [
    "한국의 4계절 특징을 간단히 설명해주세요.",
    "Python에서 리스트와 튜플의 차이점은?",
    "3 + 5 * 2 의 결과는 무엇인가요?",
]

print("📝 다양한 질문 테스트")
print("=" * 60)

for i, question in enumerate(test_questions, 1):
    print(f"\n{'─' * 60}")
    print(f"❓ 질문 {i}: {question}")
    
    messages = [
        {"role": "system", "content": "간결하게 한국어로 답변하세요."},
        {"role": "user", "content": question}
    ]
    
    start_time = time.time()
    result = pipe(messages, max_new_tokens=150, temperature=0.5, do_sample=True)
    elapsed = time.time() - start_time
    
    response_text = result[0]["generated_text"][-1]["content"]
    print(f"💬 응답: {response_text[:200]}")
    print(f"⏱️ {elapsed:.2f}초")

📝 다양한 질문 테스트

────────────────────────────────────────────────────────────
❓ 질문 1: 한국의 4계절 특징을 간단히 설명해주세요.


💬 응답: 한국은 겨울, spring(봄), summer(여름), autumn(가을)으로 구성된 4개의 계절이 있습니다:

1. 겨울: 한파와 눈이 많은 계절입니다.
2. spring: 봄날씨로 꽃이 피고 새싹이 자라납니다.
3. summer: 더운 여름날씨로, 해수욕장과 열대야가 많습니다.
4. autumn: 가을 날씨로, 단풍나무가 붉게 변합니다.

이러한 계절 
⏱️ 4.25초

────────────────────────────────────────────────────────────
❓ 질문 2: Python에서 리스트와 튜플의 차이점은?
💬 응답: Python에서 리스트(LIST)와 튜플(TUPLE)은 주로 동일한 용도를 하지만, 몇 가지 중요한 차이점이 있습니다:

1. 접근 방법:
   - 리스트: 인덱싱과 슬라이싱 가능
   - 튜플: 인덱싱과 슬라이싱 불가능

2. 메모리 사용:
   - 리스트: 더 많은 메모리를 사용
   - 튜플: 적게 메모리 사용

3. 가변성:
   - 리스트: 변경 
⏱️ 4.23초

────────────────────────────────────────────────────────────
❓ 질문 3: 3 + 5 * 2 의 결과는 무엇인가요?
💬 응답: 3 + 5 * 2의 결과는 13입니다.
⏱️ 0.43초


---

## 5️⃣ 생성 파라미터 정리

`model.generate()` / `pipeline()`이 받는 주요 인자.

| 파라미터 | 의미 | 권장값 |
|----------|------|--------|
| `max_new_tokens` | 새로 생성할 최대 토큰 수 | 50~500 |
| `do_sample` | 샘플링 여부 (False = greedy) | True |
| `temperature` | 분포 sharpness (낮을수록 결정적) | 0.7~1.0 |
| `top_p` | nucleus sampling 누적 확률 | 0.9~0.95 |
| `top_k` | 상위 K개만 후보 | 50 |
| `repetition_penalty` | 반복 억제 | 1.0~1.2 |
| `pad_token_id` | padding 토큰 ID | `tokenizer.eos_token_id` |

자세한 비교는 Session 03 "텍스트 생성 전략" 참조.

---

In [8]:
# 📖 생성 파라미터 정리
params = [
    ("max_new_tokens", "생성할 최대 토큰 수", "128~512"),
    ("temperature", "랜덤성 조절 (높을수록 다양)", "0.1~1.0"),
    ("top_p", "누적 확률 기반 샘플링", "0.9~0.95"),
    ("top_k", "상위 k개 토큰에서 샘플링", "20~50"),
    ("repetition_penalty", "반복 억제 (1.0=없음)", "1.0~1.3"),
    ("do_sample", "샘플링 활성화 (False=greedy)", "True/False"),
    ("num_beams", "빔 서치 빔 수", "1~5"),
]

print("📖 텍스트 생성 주요 파라미터")
print("=" * 65)
print(f"{'파라미터':<25} {'설명':<25} {'권장값':>12}")
print("-" * 65)
for name, desc, val in params:
    print(f"{name:<25} {desc:<25} {val:>12}")

print(f"\n💡 팁:")
print(f"  🔹 temperature=0: 항상 같은 결과 (결정적)")
print(f"  🔹 temperature=1: 다양한 결과 (창의적)")
print(f"  🔹 코드 생성: temperature=0.2, 대화: temperature=0.7")

📖 텍스트 생성 주요 파라미터
파라미터                      설명                                 권장값
-----------------------------------------------------------------
max_new_tokens            생성할 최대 토큰 수                    128~512
temperature               랜덤성 조절 (높을수록 다양)               0.1~1.0
top_p                     누적 확률 기반 샘플링                  0.9~0.95
top_k                     상위 k개 토큰에서 샘플링                   20~50
repetition_penalty        반복 억제 (1.0=없음)                 1.0~1.3
do_sample                 샘플링 활성화 (False=greedy)      True/False
num_beams                 빔 서치 빔 수                           1~5

💡 팁:
  🔹 temperature=0: 항상 같은 결과 (결정적)
  🔹 temperature=1: 다양한 결과 (창의적)
  🔹 코드 생성: temperature=0.2, 대화: temperature=0.7


In [9]:
# 🧹 Part A 메모리 정리 — Part B로 가기 전에 실행
try:
    del model, tokenizer
except NameError:
    pass
try:
    del pipe
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✅ VRAM 정리: {torch.cuda.memory_allocated() / 1024**3:.2f}GB 사용 중")

✅ VRAM 정리: 0.01GB 사용 중


---

# 🦾 Part B — TRL 라이브러리 기본

---

## 6️⃣ TRL(Transformer Reinforcement Learning) 라이브러리란?

[**TRL**](https://github.com/huggingface/trl) 은 HuggingFace가 만든 **LLM 후처리(post-training) 통합 라이브러리**입니다. SFT / DPO / GRPO / PPO 등의 학습 알고리즘을 통일된 `Trainer` 인터페이스로 제공합니다.

### TRL이 제공하는 주요 Trainer

| Trainer | 학습 방식 | 본 과정 다루는 위치 |
|---------|----------|---------------------|
| **SFTTrainer** | Supervised Fine-Tuning (지도학습) | Session 10 |
| **DPOTrainer** | Direct Preference Optimization | Session 20 (Part 3) |
| **GRPOTrainer** | Group Relative Policy Optimization | Session 21 (Part 3) |
| **PPOTrainer** | Proximal Policy Optimization | (개념만) |
| **RewardTrainer** | Reward model 학습 | (개념만) |

### 왜 TRL인가?

- 🔹 **표준 인터페이스**: HuggingFace `Trainer`와 동일한 API → 학습 곡선 일관
- 🔹 **PEFT 통합**: LoRA / QLoRA를 코드 한 줄로 적용 (`peft_config`)
- 🔹 **데이터 형식 자동 처리**: ChatML / messages 자동 토큰화
- 🔹 **활발한 업데이트**: 최신 알고리즘(GRPO 등) 빠르게 반영

---

---

## 7️⃣ TrainingArguments / SFTConfig 핵심 파라미터

학습 설정은 `TrainingArguments`(HF Trainer) 또는 그 확장인 `SFTConfig`(TRL SFTTrainer)로 지정합니다.

### 핵심 파라미터

| 파라미터 | 의미 | RTX 4060 권장값 |
|----------|------|------------------|
| `per_device_train_batch_size` | GPU 당 배치 크기 | 1~2 |
| `gradient_accumulation_steps` | 그래디언트 누적 (실효 배치 ×N) | 4~8 |
| `learning_rate` | 학습률 | 2e-4 (LoRA) / 5e-5 (FFT) |
| `num_train_epochs` | 에폭 수 | 1~3 |
| `bf16` / `fp16` | 혼합 정밀도 | bf16=True (Ampere+) |
| `gradient_checkpointing` | 메모리 절약 (느려짐) | True (VRAM 부족 시) |
| `max_seq_length` | 최대 시퀀스 길이 (SFT) | 1024~2048 |
| `logging_steps` | 로그 출력 간격 | 10~50 |
| `save_steps` | 체크포인트 저장 간격 | 100~500 |
| `output_dir` | 모델/체크포인트 저장 경로 | `./output/...` |

> 💡 **실효 배치 크기** = `per_device_train_batch_size × gradient_accumulation_steps × n_gpus`

---

In [10]:
# TrainingArguments 설명
print("📚 TrainingArguments 핵심 파라미터 설명")
print("=" * 60)

training_args_explained = {
    "output_dir": {
        "설명": "학습 결과(체크포인트, 로그) 저장 경로",
        "RTX 4060 권장값": "'./output'",
    },
    "num_train_epochs": {
        "설명": "전체 데이터를 몇 번 반복 학습할지",
        "RTX 4060 권장값": "3 (과적합 주의)",
    },
    "per_device_train_batch_size": {
        "설명": "GPU당 한 번에 처리할 데이터 수",
        "RTX 4060 권장값": "1 (VRAM 제한)",
    },
    "gradient_accumulation_steps": {
        "설명": "가상 배치 크기 = batch_size × 이 값",
        "RTX 4060 권장값": "8 (effective batch=8)",
    },
    "learning_rate": {
        "설명": "학습률 (LoRA는 높게 설정 가능)",
        "RTX 4060 권장값": "2e-4",
    },
    "warmup_ratio": {
        "설명": "학습 초반 warmup 비율",
        "RTX 4060 권장값": "0.03",
    },
    "fp16": {
        "설명": "16bit 혼합 정밀도 학습",
        "RTX 4060 권장값": "True (VRAM 절약)",
    },
    "logging_steps": {
        "설명": "몇 스텝마다 로그 출력",
        "RTX 4060 권장값": "10",
    },
    "save_strategy": {
        "설명": "체크포인트 저장 전략",
        "RTX 4060 권장값": "'epoch'",
    },
    "optim": {
        "설명": "옵티마이저 종류",
        "RTX 4060 권장값": "'adamw_torch'",
    },
}

for param, info in training_args_explained.items():
    print(f"\n  📌 {param}")
    print(f"     설명: {info['설명']}")
    print(f"     권장: {info['RTX 4060 권장값']}")

📚 TrainingArguments 핵심 파라미터 설명

  📌 output_dir
     설명: 학습 결과(체크포인트, 로그) 저장 경로
     권장: './output'

  📌 num_train_epochs
     설명: 전체 데이터를 몇 번 반복 학습할지
     권장: 3 (과적합 주의)

  📌 per_device_train_batch_size
     설명: GPU당 한 번에 처리할 데이터 수
     권장: 1 (VRAM 제한)

  📌 gradient_accumulation_steps
     설명: 가상 배치 크기 = batch_size × 이 값
     권장: 8 (effective batch=8)

  📌 learning_rate
     설명: 학습률 (LoRA는 높게 설정 가능)
     권장: 2e-4

  📌 warmup_ratio
     설명: 학습 초반 warmup 비율
     권장: 0.03

  📌 fp16
     설명: 16bit 혼합 정밀도 학습
     권장: True (VRAM 절약)

  📌 logging_steps
     설명: 몇 스텝마다 로그 출력
     권장: 10

  📌 save_strategy
     설명: 체크포인트 저장 전략
     권장: 'epoch'

  📌 optim
     설명: 옵티마이저 종류
     권장: 'adamw_torch'


---

## 8️⃣ SFTTrainer 인터페이스 미리보기

본 셀은 **코드 구조 확인용**입니다. 실제 학습은 Session 12에서 실행합니다.

### SFTTrainer 기본 사용 패턴

```python
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig

# 1) 학습 설정
sft_config = SFTConfig(
    output_dir="./output/sft_qwen",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    bf16=True,
    max_seq_length=1024,
    gradient_checkpointing=True,
    logging_steps=10,
)

# 2) LoRA 설정 (PEFT 적용)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

# 3) SFTTrainer 구성
trainer = SFTTrainer(
    model=model,                    # 사전학습 모델
    tokenizer=tokenizer,
    train_dataset=train_dataset,    # ChatML 형식 데이터셋
    args=sft_config,
    peft_config=peft_config,        # LoRA 자동 적용
)

# 4) 학습 실행
trainer.train()

# 5) 어댑터 저장
trainer.save_model("./output/sft_qwen/final_adapter")
```

### 🔑 기억할 점

1. **모델은 그대로** — `model`을 받아 LoRA만 입혀 학습 (PEFT 통합)
2. **데이터 형식 자유** — `messages` 형식 / `text` 컬럼 / `instruction-output` 등 자동 처리
3. **Loss는 NTP** — SFT 본질은 next token prediction, 응답 부분 토큰의 cross-entropy
4. **저장은 어댑터만** — LoRA 가중치(보통 ~수십 MB)만 저장, base model은 별도

---

---

## 📝 정리 및 핵심 요약

### 본 세션에서 익힌 도구

| 라이브러리 | 핵심 클래스 / 함수 | 본 과정 활용 |
|------------|---------------------|---------------|
| `transformers` | `AutoTokenizer`, `AutoModelForCausalLM`, `pipeline` | 모든 노트북 |
| `transformers` | `model.generate(**inputs, ...)` | 추론 / 평가 |
| `trl` | `SFTTrainer`, `SFTConfig` | Session 10 (SFT) |
| `trl` | `DPOTrainer` | Session 20 |
| `trl` | `GRPOTrainer` | Session 21 |
| `peft` | `LoraConfig`, `get_peft_model` | Session 08, 09 |

### 학습 흐름 매핑

```
Session 06   파인튜닝 개념 정리 (이론)
   ↓
Session 07  도구 기본 (← 지금 여기) — Transformers + TRL 인터페이스 숙지
   ↓
Session 08~09  LoRA 이론 + 실전
   ↓
Session 13   데이터 파이프라인 & 합성 데이터
   ↓
Session 10   SFT 실습 — 위 도구·데이터로 본격 학습
```

### 다음 세션 예고

- 🔜 **Session 08**: LoRA vs Full Fine-tuning 이론 비교
- 🔜 **Session 09**: LoRA vs FFT 실전 비교 실습
- 🔜 **Session 12c**: Unsloth 기반 파인튜닝 (LoRA 가속)
- 🔜 **Session 13**: 데이터 파이프라인 & 합성 데이터/Distillation
- 🔜 **Session 10**: 본격 SFT 실습 — TRL `SFTTrainer` + LoRA + Qwen2.5-1.5B

---

### 💡 실습 과제

1. 본 노트북의 `pipeline()` 코드로 본인의 질문 3개를 던져보고 결과 비교
2. `temperature` 값을 0.3 / 0.7 / 1.2로 바꿔 같은 질문에 어떻게 달라지는지 관찰
3. (선택) HuggingFace Hub에서 다른 모델(`LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct` 등) 로딩해 응답 차이 비교

---

In [11]:
print("✅ Session 07 완료!")
print("📚 다음 세션: 11b 데이터 파이프라인 & 합성 데이터 → 12 SFT 실습")

✅ Session 07 완료!
📚 다음 세션: 11b 데이터 파이프라인 & 합성 데이터 → 12 SFT 실습
